In [15]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo 
from matplotlib import pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from itertools import combinations_with_replacement

Polish bankruptcy dataset (id=365)

* Parameters: 64 continuous financial measurements (after dropping `year`)
* Number observations: 43,405

AFTER CLEANING:
* Parameters: 63
* Number observations: 40,506

Target: whether or not the firm went bankrupt

In [16]:
# fetch dataset
data_id=365
data_frame = fetch_ucirepo(id=data_id) 
  
# data (as pandas dataframes) 
X = data_frame.data.features
y = data_frame.data.targets
df = pd.concat([X, y], axis=1)
df = df.drop(columns=['year','A36'])
df = df.to_numpy()
M = df[:, :-1]
y = df[:, -1]

There are a lot of NaNs in this dataset. First, I automatically remove any observation (row) with 10 or more NaNs. Then, I remove any parameter (column) with 1,000 or more NaNs. Finally, I remove all other observations (rows) which have 1 or more NaNs. Per the UCI website, there are no NaNs in the `y` column.

In [17]:
# Cleaning, get rid of NaNs

a,b=np.where(np.isnan(M))
values, counts = np.unique(a, return_counts=True)
rows_delete = values[counts>=10]
M = np.delete(M, rows_delete, axis=0)
y = np.delete(y, rows_delete)
values, counts = np.unique(b, return_counts=True)
cols_delete = values[counts>=1000]
M = np.delete(M, cols_delete, axis=1)
a, b = np.where(np.isnan(M))
rows_delete = np.unique(a)
M = np.delete(M, rows_delete, axis=0)
y = np.delete(y,rows_delete)
num_obs, params = M.shape

Artificially adding parameters

In [18]:
for i,j in combinations_with_replacement(range(params), 2):
    M = np.concatenate((M, (M[:,i]*M[:,j]).reshape(-1, 1)), axis=1)
for i,j,k in combinations_with_replacement(range(params), 3):
    M = np.concatenate((M, (M[:,i]*M[:,j]*M[:,k]).reshape(-1, 1)), axis=1)

num_obs, params = M.shape
num_obs, params

KeyboardInterrupt: 

In [ ]:
rng=np.random.default_rng(51)

def resort_data(num_pts, num_params, M, y):
    '''
    num_pts: How many data points are needed (size of training data)
    M: Entire data matrix (including y-values)'''

    # Reorder M by observation so that logit doesn't break
    y_T = np.zeros(num_pts)
    while not np.any(y_T != y_T[0]):
        indices = rng.choice(np.arange(num_obs), size=num_pts, replace=False)
        y_T = y[indices]

    M_new = np.concatenate((M[indices, :], np.delete(M, indices, axis=0)))

    # Randomly sort parameters
    param_ordering = rng.permutation(params)
    M_new = M_new[:, param_ordering]

    # Breaking down the resorted matrix
    M_TM = M_new[:num_pts, :num_params]
    M_TU = M_new[:num_pts, num_params:]
    M_PM = M_new[num_pts:, :num_params]
    M_PU = M_new[num_pts:, num_params:]
    y_P = np.delete(y, indices)
    y_new = np.concatenate((y_T, y_P))
    return M_new, M_TM, M_TU, M_PM, M_PU, y_T, y_P, y_new

Since you can only compare AIC values for models trained on the same dataset, I am "pre-generating" the `M_TM`, `M_TU`, `M_PM`, `M_PU` blocks to be used, 10 for each combination of model dimension and size.

In [ ]:
error_matrix = []
aic_matrix=[]

for i in range(2,num_obs,500):
    datasets=[resort_data(i, j, M, y) for _ in range(10)]
    for j in range(1,params,500):
        
        



# Compute log loss (risk)
for j in range(2,params,obs_step_size):
    print(j)
    error_matrix.append([])  
    aic_matrix.append([])
    for i in range(1,params):
        listy = []
        model=LogisticRegression(C=np.inf,solver='lbfgs')
        M_new, M_TM, M_TU, M_PM, M_PU, y_T, y_P, y_new = resort_data(j, i, M, y)
        model.fit(M_TM, y_T)
        y_hat = model.predict_proba(np.concatenate((M_TM,M_PM)))[:, 1]
        error = log_loss(y_new, y_hat, labels=[0,1])
        error_matrix[-1].append(error)
    
matrices = np.array(error_matrix)

# Plotting it
fig, ax = plt.subplots(1,1,figsize=(6.5,6))

# Clip to 95th percentile
p95 = np.percentile(matrices, 95)
matrices = np.clip(matrices, a_min=0, a_max=p95)

heatmap=ax.imshow(matrices, aspect='auto', origin='lower', extent = [1,params,2,num_obs])
plt.xlabel("Number of Parameters")
plt.ylabel("Number of Data Points")
fig.colorbar(heatmap)
plt.plot([1,min(num_obs,params)],[1,min(num_obs,params)],'--',color='red')
plt.ylim((2,max))
plt.xlim((1,params))
plt.title('Cross Entropy Loss for Polish Bankruptcy (365) w/ AIC')

plt.tight_layout()
plt.savefig(f'logit_risk_{data_id}_aic.png',dpi=400)
plt.show()